In [1]:
from langchain_mistralai import MistralAIEmbeddings
from langgraph.store.memory import InMemoryStore

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
embedding = MistralAIEmbeddings(model="mistral-embed")
store = InMemoryStore(index={
    "embed": embedding,
    "dimension": 1024
})

In [3]:
namespace = ('user','u1')

In [4]:
store.put(namespace, "1", {"data": "I am a React and Node.js developer"})
store.put(namespace, "2", {"data": "I am learning system design"})
store.put(namespace, "3", {"data": "I am learning LangGraph"})
store.put(namespace, "4", {"data": "I prefer short and concise explanations"})
store.put(namespace, "5", {"data": "I am building a LangGraph chatbot"})
store.put(namespace, "6", {"data": "I want to learn long-term memory"})
store.put(namespace, "7", {"data": "I use Manjaro Linux"})
store.put(namespace, "8", {"data": "I am preparing for senior developer interviews"})

In [5]:
store.get(namespace,"2")

Item(namespace=['user', 'u1'], key='2', value={'data': 'I am learning system design'}, created_at='2026-10-04T11:20:38.080519+00:00', updated_at='2026-10-04T11:20:38.080528+00:00')

In [6]:
store.search(namespace)

[Item(namespace=['user', 'u1'], key='1', value={'data': 'I am a React and Node.js developer'}, created_at='2026-10-04T11:20:37.558040+00:00', updated_at='2026-10-04T11:20:37.558053+00:00', score=None),
 Item(namespace=['user', 'u1'], key='2', value={'data': 'I am learning system design'}, created_at='2026-10-04T11:20:38.080519+00:00', updated_at='2026-10-04T11:20:38.080528+00:00', score=None),
 Item(namespace=['user', 'u1'], key='3', value={'data': 'I am learning LangGraph'}, created_at='2026-10-04T11:20:38.421960+00:00', updated_at='2026-10-04T11:20:38.421967+00:00', score=None),
 Item(namespace=['user', 'u1'], key='4', value={'data': 'I prefer short and concise explanations'}, created_at='2026-10-04T11:20:38.772766+00:00', updated_at='2026-10-04T11:20:38.772774+00:00', score=None),
 Item(namespace=['user', 'u1'], key='5', value={'data': 'I am building a LangGraph chatbot'}, created_at='2026-10-04T11:20:39.113517+00:00', updated_at='2026-10-04T11:20:39.113523+00:00', score=None),
 Ite

In [7]:
# store.search(namespace,query='what user currently learning',limit=2)
store.search(namespace,query='What technologies do I work with?',limit=2)

[Item(namespace=['user', 'u1'], key='1', value={'data': 'I am a React and Node.js developer'}, created_at='2026-10-04T11:20:37.558040+00:00', updated_at='2026-10-04T11:20:37.558053+00:00', score=0.7045669445218763),
 Item(namespace=['user', 'u1'], key='2', value={'data': 'I am learning system design'}, created_at='2026-10-04T11:20:38.080519+00:00', updated_at='2026-10-04T11:20:38.080528+00:00', score=0.6891768372380558)]

In [8]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq

In [9]:
class State(TypedDict):
    query: str
    user_memory: Annotated[str, "Memory"] = ""
    response: Annotated[str, "AI Response"] = ""

In [10]:
llm = ChatGroq(model='openai/gpt-oss-120b',temperature=0.1)

SYSTEM_PROMPT = ''' 
You are a concise chatbot.

Rules:
- Answer in 1-2 short sentences only.
- Maximum 30 words per response.
'''

In [11]:
def seach_in_memory(state,config,store):
    user_id = config['configurable']['user_id']
    namespace = ('user',user_id)
    query = state['query']
    print(namespace)
    items = store.search(namespace,query=query,limit=2)

    memory = "\n".join(f"- {item.value.get('data','')}" for item in items)

    print('Memory',memory)

    return {
        "user_memory": memory
    }

def chat_node(state):
    user_memory = state["user_memory"]

    query = f"""
{SYSTEM_PROMPT}

Relevant memory:
{user_memory}

Query:
{state['query']}
"""

    response = llm.invoke(query)

    return {
        "response": response.content
    }

In [12]:
graph = StateGraph(State)

graph.add_node('seach_in_memory',seach_in_memory)
graph.add_node('chat_node', chat_node)

graph.add_edge(START,'seach_in_memory')

graph.add_edge('seach_in_memory','chat_node')
graph.add_edge('chat_node',END)

workflow = graph.compile(store=store)

In [13]:
workflow.invoke({'query': 'What technologies do I work with?'}, config={'configurable': {'user_id': 'u1'}})

('user', 'u1')
Memory - I am a React and Node.js developer
- I am learning system design


{'query': 'What technologies do I work with?',
 'user_memory': '- I am a React and Node.js developer\n- I am learning system design',
 'response': 'You work with React for the frontend and Node.js for the backend, and you’re also studying system‑design concepts.'}